In [1]:
import numpy as np
import open3d as o3d

def flip_point_cloud(point_cloud, axis='x'):
    if axis == 'x':
        rotation_matrix = np.array([[1,  0,  0],
                                    [0, -1,  0],
                                    [0,  0, -1]])
    elif axis == 'y':
        rotation_matrix = np.array([[-1,  0,  0],
                                    [0,  1,  0],
                                    [0,  0, -1]])
    elif axis == 'z':
        rotation_matrix = np.array([[-1,  0,  0],
                                    [0, -1,  0],
                                    [0,  0,  1]])
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(point_cloud)
    pcd.rotate(rotation_matrix)
    return np.asarray(pcd.points)

# 讀取原始點雲數據
# ply_file_path = 'C:/Users/mmdl/Desktop/tube_ply/1209_3.ply' 
ply_file_path = 'C:/Users/mmdl/Desktop/tube_ply/5_0530_1.ply' 
# ply_file_path = 'C:/Users/mmdl/Desktop/point_test/point_cloud_0000018.ply'
pcd = o3d.io.read_point_cloud(ply_file_path)
point_cloud = np.asarray(pcd.points)

if pcd.has_colors():
    original_colors = np.asarray(pcd.colors)
else:
    original_colors = np.ones_like(point_cloud)

# 翻轉點雲
flipped_point_cloud = flip_point_cloud(point_cloud, axis='x')

pcd_original = o3d.geometry.PointCloud()
pcd_original.points = o3d.utility.Vector3dVector(point_cloud)
pcd_original.paint_uniform_color([0, 0, 1])  # 原始点云设为蓝色

pcd_flipped = o3d.geometry.PointCloud()
pcd_flipped.points = o3d.utility.Vector3dVector(flipped_point_cloud)
pcd_flipped.colors = o3d.utility.Vector3dVector(original_colors)  # 使用原始颜色设置翻转后的点云

vis = o3d.visualization.Visualizer()
vis.create_window()
# vis.add_geometry(pcd_original)
vis.add_geometry(pcd_flipped)

opt = vis.get_render_option()
vis.run()
vis.destroy_window()

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:
from sklearn.cluster import DBSCAN

# 離群點剃除
# pcd_flipped, ind = pcd_flipped.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)

# 可視化離群點剔除後的結果
o3d.visualization.draw_geometries([pcd_flipped], window_name="離群點剃除後的點雲",
                                  width=800, height=600,
                                  left=50, top=50,
                                  point_show_normal=False, mesh_show_wireframe=False,
                                  mesh_show_back_face=False)
# 計算法向量
pcd_flipped.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
# pcd_flipped.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))

# Poisson 表面重建
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd_flipped, depth=8)

# 可視化Poisson重建結果
o3d.visualization.draw_geometries([mesh], window_name="Poisson表面重建",
                                  width=800, height=600,
                                  left=50, top=50,
                                  point_show_normal=False, mesh_show_wireframe=True,
                                  mesh_show_back_face=False)
# 過濾低密度網格
densities = np.asarray(densities)
vertices_to_remove = densities < np.quantile(densities, 0.07)
mesh.remove_vertices_by_mask(vertices_to_remove)

o3d.visualization.draw_geometries([mesh], window_name="過濾後的表面重建",
                                  width=800, height=600,
                                  left=50, top=50,
                                  point_show_normal=False, mesh_show_wireframe=True,
                                  mesh_show_back_face=False)
#-----------------------------------------------------------------------------------------------------------------------------

# 從重建後的表面網格中均勻採樣點雲
sampled_pcd = mesh.sample_points_uniformly(number_of_points=500000)
points = np.asarray(sampled_pcd.points)

# DBSCAN 
dbscan = DBSCAN(eps=1, min_samples=50, algorithm='ball_tree')  # 根據需要調整 eps 和 min_samples 參數
labels = dbscan.fit_predict(points)

unique_labels = set(labels)
main_cloud_label = max(unique_labels, key=list(labels).count)  # 找到數量最多的標籤
main_points = points[labels == main_cloud_label]

main_pcd = o3d.geometry.PointCloud()
main_pcd.points = o3d.utility.Vector3dVector(main_points)

original_colors = np.asarray(sampled_pcd.colors)
main_colors = original_colors[labels == main_cloud_label]
main_pcd.colors = o3d.utility.Vector3dVector(main_colors)

o3d.visualization.draw_geometries([main_pcd], window_name="主要點雲",
                                  width=800, height=600,
                                  left=50, top=50,
                                  point_show_normal=False, mesh_show_wireframe=False,
                                  mesh_show_back_face=False)

flipped_ply_file_path = 'C:/Users/mmdl/Desktop/test_data/5_0530_1.ply'
o3d.io.write_point_cloud(flipped_ply_file_path, main_pcd)